In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.neighbors import LocalOutlierFactor
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_samples

# -------------------------------------------------
# 0) GitHub-friendly paths 
# -------------------------------------------------
DATA_DIR = Path("data")
OUT_DIR = Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

input_file = DATA_DIR / "FIFA_2017_2022_COMBINED.csv"
output_file = OUT_DIR / "FIFA_2017_LOF_CLUSTER_POSITION.csv"

# -------------------------------------------------
# 1) Load combined FIFA file
# -------------------------------------------------
df = pd.read_csv(input_file)
df.columns = [c.strip() for c in df.columns]

# -------------------------------------------------
# 2) Filter ONLY FIFA 2017
# -------------------------------------------------
df["fifa_Year"] = df["fifa_Year"].astype(int)
df_2017 = df[df["fifa_Year"] == 2017].copy().reset_index(drop=True)

print("Rows in FIFA 2017:", len(df_2017))

# -------------------------------------------------
# 3) Create FIFA summary stats (from attributes)
# -------------------------------------------------
def safe_mean(frame, cols):
    return frame[cols].apply(pd.to_numeric, errors="coerce").mean(axis=1)

df_2017["Pace"] = safe_mean(df_2017, ["Acceleration", "SprintSpeed"])
df_2017["Shooting"] = safe_mean(df_2017, ["Finishing", "ShotPower", "LongShots", "Volleys", "Penalties"])
df_2017["Passing"] = safe_mean(df_2017, ["ShortPassing", "LongPassing", "Vision", "Crossing", "Curve", "FKAccuracy"])
df_2017["Defending"] = safe_mean(df_2017, ["Interceptions", "Marking", "StandingTackle", "SlidingTackle"])
df_2017["Physicality"] = safe_mean(df_2017, ["Strength", "Stamina", "Aggression", "Jumping", "Balance"])

# -------------------------------------------------
# 4) Feature matrix (numeric only)
# -------------------------------------------------
features = [
    "Overall", "Potential",
    "Pace", "Shooting", "Passing",
    "Dribbling", "Defending", "Physicality"
]

X = df_2017[features].apply(pd.to_numeric, errors="coerce")
mask = X.notna().all(axis=1)
df_2017 = df_2017.loc[mask].copy()
X = X.loc[mask].copy()

# -------------------------------------------------
# 5) LOF (Manhattan distance)
# -------------------------------------------------
n_neighbors = 20
lof = LocalOutlierFactor(n_neighbors=n_neighbors, metric="manhattan")
_ = lof.fit_predict(X)

df_2017["LOF_score"] = -lof.negative_outlier_factor_
df_2017["LOF_neighbors"] = n_neighbors

def lof_bucket(score):
    if score <= 1.1: return "Normal"
    if score <= 1.3: return "Slight"
    if score <= 1.6: return "Moderate"
    if score <= 2.0: return "Strong"
    return "Extreme"

df_2017["LOF_bucket"] = df_2017["LOF_score"].apply(lof_bucket)

# -------------------------------------------------
# 6) LOF POSITION (rank descending)
# -------------------------------------------------
df_2017["LOF_rank_desc"] = (
    df_2017["LOF_score"]
    .rank(ascending=False, method="dense")
    .astype(int)
)

df_2017["Label_Top10_Outliers"] = np.where(
    df_2017["LOF_rank_desc"] <= 10,
    "Top 10 Outliers",
    ""
)

# -------------------------------------------------
# 7) K-Means clustering + Silhouette
# -------------------------------------------------
k = 6  # typical choice for player archetypes
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
df_2017["Cluster"] = kmeans.fit_predict(X)

df_2017["Silhouette"] = silhouette_samples(X, df_2017["Cluster"])

# -------------------------------------------------
# 8) Save Tableau-ready output
# -------------------------------------------------
df_2017.to_csv(output_file, index=False)

print("Saved:", output_file)